## 1st Step - Read results

In [ ]:
require("data.table")
require("yaml")

# List the folders with the results (WFE16v2...)
folders <- list.files("/content/buckets/b1/exp", pattern = "^WFE16v2", full.names = TRUE)
# Exclude the orchestrator outputs
folders <- folders[!grepl("_orchestrator_output$", folders)]

# Read the data
data <- rbindlist(lapply(folders, function(folder) {
  yml_path <- file.path(folder, "PARAM.yml")
  if (file.exists(yml_path)) {
    p <- read_yaml(yml_path)
    return(data.table(
      experimento = p$experimento,
      variante    = p$variant$name,
      semilla     = as.character(p$semilla_primigenia),
      ganancia    = p$resultado$ganancia_suavizada_max
    ))
  }
  return(NULL)
}))

# Pivot, seeds in rows, variants in columns
comparative_tb <- dcast(data, semilla ~ variante, value.var = "ganancia")
print(comparative_tb)

## 2nd Step - Wilcoxon Test

In [ ]:
# Variables to compare against baseline
variants_to_test <- c("without_id", "without_month_snapshot", "without_both")

test_results <- rbindlist(lapply(variants_to_test, function(var_name) {
  
  # Paired bilateral test
  wt <- wilcox.test(
    x = comparative_tb[[var_name]],
    y = comparative_tb[["baseline"]],
    paired = TRUE,
    exact = FALSE # avoid warnings on tie
  )
  
  # Difference by seed
  dif <- comparative_tb[[var_name]] - comparative_tb[["baseline"]]
  
  data.table(
    Comparacion        = paste(var_name, "vs Baseline"),
    Gana_Candidata     = sum(dif > 0),
    Gana_Baseline      = sum(dif < 0),
    Empates            = sum(dif == 0),
    Mediana_Diferencia = median(dif),
    Media_Diferencia   = mean(dif),
    P_Valor            = wt$p.value,
    Conclusion         = ifelse(wt$p.value < 0.05, 
                                "Diferencia Significativa (Rechaza H0)", 
                                "No concluyente (No se puede descartar azar)")
  )
}))

print(test_results)

## Hyperparameters Analysis V2

In [ ]:
require("data.table")
require("yaml")

# Define the edges of the grid search used
edges <- list(
  num_leaves       = c(min = 64, max = 512),
  min_data_in_leaf = c(min = 64, max = 2048),
  feature_fraction = c(min = 0.5, max = 0.8)
)

# List the folders with the results (WFE16v2...)
folders <- list.files("/content/buckets/b1/exp", pattern = "^WFE16v2", full.names = TRUE)
# Exclude the orchestrator outputs
folders <- folders[!grepl("_orchestrator_output$", folders)]

# Hyperparameters extraction from PARAM.yml
optimal_tb <- rbindlist(lapply(folders, function(folder) {
  yml_path <- file.path(folder, "PARAM.yml")
  if (file.exists(yml_path)) {
    p <- read_yaml(yml_path)

    # Winning Hyperparameters
    bests <- p$out$lgbm$mejores_hiperparametros
    
    nl <- as.numeric(bests$num_leaves)
    ml <- as.numeric(bests$min_data_in_leaf)
    ff <- as.numeric(bests$feature_fraction)
    ni <- as.numeric(bests$num_iterations)
    
    # Edges check
    toca_nl <- ifelse(nl == edges$num_leaves["min"], "Borde Min (64)",
               ifelse(nl == edges$num_leaves["max"], "Borde Max (512)", "Interior"))
    
    toca_ml <- ifelse(ml == edges$min_data_in_leaf["min"], "Borde Min (64)",
               ifelse(ml == edges$min_data_in_leaf["max"], "Borde Max (2048)", "Interior"))
    
    toca_ff <- ifelse(ff == edges$feature_fraction["min"], "Borde Min (0.5)",
               ifelse(ff == edges$feature_fraction["max"], "Borde Max (0.8)", "Interior"))
    
    return(data.table(
      experimento       = p$experimento,
      variante          = p$variant$name,
      semilla           = p$semilla_primigenia,
      num_leaves        = nl,
      min_data_in_leaf  = ml,
      feature_fraction  = ff,
      num_iterations    = ni,
      borde_nl          = toca_nl,
      borde_ml          = toca_ml,
      borde_ff          = toca_ff,
      algun_borde       = (toca_nl != "Interior" | toca_ml != "Interior" | toca_ff != "Interior")
    ))
  }
  return(NULL)
}))

# Summary: selected values distribution
cat("=== FRECUENCIA DE num_leaves (edges: 64 y 512) ===\n")
print(optimal_tb[, .N, by = .(num_leaves, borde_nl)][order(num_leaves)])

cat("\n=== FRECUENCIA DE min_data_in_leaf (edges: 64 y 2048) ===\n")
print(optimal_tb[, .N, by = .(min_data_in_leaf, borde_ml)][order(min_data_in_leaf)])

cat("\n=== FRECUENCIA DE feature_fraction (edges: 0.5 y 0.8) ===\n")
print(optimal_tb[, .N, by = .(feature_fraction, borde_ff)][order(feature_fraction)])

# List runs which end up in the edges
edge_runs <- optimal_tb[algun_borde == TRUE, .(experimento, variante, semilla, num_leaves, borde_nl, min_data_in_leaf, borde_ml, feature_fraction, borde_ff)]

cat("\n=== TOTAL DE CORRIDAS QUE TOCARON AL MENOS UN BORDE ===", nrow(edge_runs), "de", nrow(optimal_tb), "\n")
if (nrow(edge_runs) > 0) {
  print(edge_runs)
}